# 🔄 Notebook 2: File Sync & Conflict Resolution

In Notebook 1 we uploaded files into our Dropbox-like system. But the *real* challenge of a cloud file service isn't storage — it's **keeping every device in sync**. When Alice edits a file on her laptop and then opens her phone, she expects to see the latest version *instantly*. When Alice and Bob both edit the same document at the same time, the system must detect the **conflict** and handle it gracefully.

This notebook walks through how Dropbox-style sync actually works, step by step.

## Learning Objectives

By the end of this notebook, you'll understand:
- Why file synchronization across devices is a hard problem
- How a **sync_events** table tracks every file mutation
- How clients **poll** for changes (`GET /changes?since=<event_id>`)
- How **Redis Pub/Sub** enables real-time push notifications
- How to combine polling + push into a **hybrid sync** approach
- How to **detect conflicts** using file version numbers
- Two conflict resolution strategies: *last write wins* and *keep both copies*
- How a full sync cycle works end to end

## 🛠️ Setup

Start the infrastructure first:

```bash
cd 06-system-designs/dropbox
docker compose up -d
```

### Visualization Tools

- **Adminer** (PostgreSQL GUI): http://localhost:8080  
  Login: System `PostgreSQL`, Server `postgres`, User `demo`, Password `demo`, Database `dropbox_demo`
- **RedisInsight** (Redis GUI): http://localhost:5540  
  Click "Add Redis Database" → Host `redis`, Port `6379`
- **MinIO Console** (S3 GUI): http://localhost:9001  
  Login: User `minioadmin`, Password `minioadmin`

### Kernel Selection

Select the `.venv` kernel in VS Code's kernel picker (top-right of notebook).  
If it doesn't appear, reload the window: `Cmd+Shift+P` → "Reload Window".

In [ ]:
import psycopg2
import psycopg2.extras
import redis
import boto3
import json
import time
import threading
from datetime import datetime, timedelta

# ── Connection settings ────────────────────────────────────

DB_CONFIG = {
    "host": "localhost",
    "port": 5432,
    "database": "dropbox_demo",
    "user": "demo",
    "password": "demo",
}

REDIS_CONFIG = {
    "host": "localhost",
    "port": 6379,
    "decode_responses": True,
}

S3_CONFIG = {
    "endpoint_url": "http://localhost:9000",
    "aws_access_key_id": "minioadmin",
    "aws_secret_access_key": "minioadmin",
}

BUCKET = "dropbox-files"

def get_db():
    """Return a new Postgres connection."""
    return psycopg2.connect(**DB_CONFIG)

def get_redis():
    """Return a Redis client."""
    return redis.Redis(**REDIS_CONFIG)

def get_s3():
    """Return a boto3 S3 client pointed at MinIO."""
    return boto3.client("s3", **S3_CONFIG)

# ── Test connections ───────────────────────────────────────

try:
    conn = get_db()
    conn.close()
    print("✅ Connected to PostgreSQL")
except Exception as e:
    print(f"❌ PostgreSQL failed: {e}")
    print("   Run: docker compose up -d")

try:
    r = get_redis()
    r.ping()
    print("✅ Connected to Redis")
except Exception as e:
    print(f"❌ Redis failed: {e}")
    print("   Run: docker compose up -d")

try:
    s3 = get_s3()
    s3.head_bucket(Bucket=BUCKET)
    print("✅ Connected to MinIO (S3)")
except Exception as e:
    print(f"❌ MinIO failed: {e}")
    print("   Run: docker compose up -d")

## 🧹 Reset Lab State

We'll clean up leftover data from previous notebook runs so we start fresh.

In [ ]:
conn = get_db()
conn.autocommit = True
cur = conn.cursor()

# Delete in dependency order
cur.execute("DELETE FROM sync_events")
cur.execute("DELETE FROM shared_files")
cur.execute("DELETE FROM chunks")
cur.execute("DELETE FROM files")

cur.close()
conn.close()

# Flush Redis keys from previous runs
r = get_redis()
for key in r.scan_iter("sync:*"):
    r.delete(key)

print("✅ Lab state reset — tables cleaned, Redis flushed")

---

## 1️⃣ Why Sync Is Hard

Imagine Alice is on an airplane with no Wi-Fi. She edits `report.txt` on her laptop.  
Meanwhile, Bob is at the office editing the **same file** on his desktop.  
When Alice lands and reconnects, the system sees **two different versions** of `report.txt`.

This is the fundamental sync problem. Three things make it hard:

| Challenge | Why it's hard |
|---|---|
| **Multiple devices** | Alice has a laptop, phone, and tablet — all need the same files |
| **Offline edits** | A device can change files while disconnected from the internet |
| **Network partitions** | Even when online, messages can be delayed, duplicated, or lost |

### The analogy

Think of it like two people editing the same Google Doc **on an airplane** (no internet).  
When they both land, Google has to figure out: whose version is "right"?  
Or maybe *both* versions contain important changes that need to be merged.

Dropbox solves this with three building blocks:
1. **Event log** — record every change that happens
2. **Sync protocol** — tell devices about changes (polling + push)
3. **Conflict resolution** — decide what to do when two edits collide

Let's build each one. 👇

---

## 2️⃣ The `sync_events` Table — Recording Every Change

Every time a file is created, updated, deleted, or shared, we insert a row into `sync_events`.  
This table is the **source of truth** for what has changed.

```
sync_events
┌────┬─────────┬─────────┬────────────┬─────────────────────┐
│ id │ user_id │ file_id │ event_type │ created_at          │
├────┼─────────┼─────────┼────────────┼─────────────────────┤
│  1 │       1 │      10 │ created    │ 2025-01-01 10:00:00 │
│  2 │       1 │      10 │ updated    │ 2025-01-01 10:05:00 │
│  3 │       2 │      11 │ created    │ 2025-01-01 10:06:00 │
└────┴─────────┴─────────┴────────────┴─────────────────────┘
```

A client can ask: *"What changed since event #1?"* and get back events #2 and #3.  
This is called a **change feed** (or **event log**).

Let's create some files and see the events appear.

In [ ]:
def create_file(owner_id, file_name, content, mime_type="text/plain"):
    """
    Create a file in the system:
      1. Upload bytes to MinIO (object storage)
      2. Insert metadata row into Postgres
      3. Insert a 'created' sync event
    Returns the new file's id.
    """
    import hashlib

    data = content.encode("utf-8") if isinstance(content, str) else content
    fingerprint = hashlib.sha256(data).hexdigest()
    storage_key = f"user_{owner_id}/{file_name}"

    # 1. Upload to MinIO
    s3 = get_s3()
    s3.put_object(Bucket=BUCKET, Key=storage_key, Body=data, ContentType=mime_type)

    # 2. Insert file metadata
    conn = get_db()
    conn.autocommit = True
    cur = conn.cursor()
    cur.execute("""
        INSERT INTO files (owner_id, file_name, mime_type, file_size, fingerprint,
                           storage_key, status, version)
        VALUES (%s, %s, %s, %s, %s, %s, 'uploaded', 1)
        RETURNING id
    """, (owner_id, file_name, mime_type, len(data), fingerprint, storage_key))
    file_id = cur.fetchone()[0]

    # 3. Record the sync event
    cur.execute("""
        INSERT INTO sync_events (user_id, file_id, event_type)
        VALUES (%s, %s, 'created')
    """, (owner_id, file_id))

    cur.close()
    conn.close()
    return file_id


# Alice creates two files
alice_id = 1
f1 = create_file(alice_id, "report.txt", "Q1 sales were strong.")
f2 = create_file(alice_id, "notes.txt",  "Meeting notes from Monday.")

print(f"✅ Alice created report.txt  (file_id={f1})")
print(f"✅ Alice created notes.txt   (file_id={f2})")

In [ ]:
# Let's look at the sync_events table

conn = get_db()
cur = conn.cursor(cursor_factory=psycopg2.extras.RealDictCursor)
cur.execute("""
    SELECT se.id, u.username, f.file_name, se.event_type, se.created_at
    FROM sync_events se
    JOIN users u  ON u.id = se.user_id
    JOIN files f  ON f.id = se.file_id
    ORDER BY se.id
""")
rows = cur.fetchall()
cur.close()
conn.close()

print("📋 sync_events table:")
print(f"{'ID':>4}  {'User':<10} {'File':<20} {'Event':<10} {'When'}")
print("-" * 70)
for row in rows:
    print(f"{row['id']:>4}  {row['username']:<10} {row['file_name']:<20} "
          f"{row['event_type']:<10} {row['created_at']}")

print(f"\n💡 Every file creation produced exactly one sync event.")

---

## 3️⃣ Polling for Changes

The simplest way for a device to stay up-to-date is **polling**: the client periodically
asks the server *"what changed since I last checked?"*

```
Device A (laptop)                       Server
    │                                     │
    │─── GET /changes?since=0 ──────────▶│  "give me everything"
    │◀── [{id:1, created, report.txt},   │
    │     {id:2, created, notes.txt}] ───│
    │                                     │
    │     ... 30 seconds later ...        │
    │                                     │
    │─── GET /changes?since=2 ──────────▶│  "anything new after event #2?"
    │◀── [] ─────────────────────────────│  "nope, nothing new"
```

The client remembers the **last event id** it saw and uses that as the `since` cursor.

Let's build this.

In [ ]:
def poll_changes(user_id, since_event_id=0):
    """
    Simulate: GET /files/changes?since=<event_id>

    Returns every sync_event newer than since_event_id that touches a file this
    user can see — one they own, or one that has been shared with them.

    Note what we deliberately do NOT filter on: `sync_events.user_id`, which
    records who *caused* the change. A change feed has to answer "what changed
    for me", not "what did I change" — filter on the actor and Bob would never
    hear about Alice's edits to a file she shared with him.
    """
    conn = get_db()
    cur = conn.cursor(cursor_factory=psycopg2.extras.RealDictCursor)
    cur.execute("""
        SELECT se.id AS event_id, f.file_name, se.event_type,
               f.version, se.created_at
        FROM sync_events se
        JOIN files f ON f.id = se.file_id
        LEFT JOIN shared_files sf
               ON sf.file_id = f.id AND sf.shared_with = %s
        WHERE se.id > %s
          AND (f.owner_id = %s OR sf.id IS NOT NULL)
        ORDER BY se.id
    """, (user_id, since_event_id, user_id))
    events = cur.fetchall()
    cur.close()
    conn.close()
    return events


# ── Scenario: Alice's phone syncs for the first time ──────

print("📱 Alice's phone polls for the first time (since=0):")
events = poll_changes(user_id=alice_id, since_event_id=0)

for e in events:
    print(f"   Event #{e['event_id']}: {e['event_type']} → {e['file_name']} (v{e['version']})")

last_seen = events[-1]["event_id"] if events else 0
print(f"\n   Phone now remembers: last_seen_event = {last_seen}")

assert len(events) == 2, f"expected the two 'created' events, got {len(events)}"
assert {e["event_type"] for e in events} == {"created"}

In [ ]:
# ── Alice edits report.txt on her laptop ──────────────────

def update_file(file_id, owner_id, new_content):
    """
    Update a file:
      1. Overwrite bytes in MinIO
      2. Bump the version number
      3. Record an 'updated' sync event
    Returns the new version number.
    """
    import hashlib

    data = new_content.encode("utf-8") if isinstance(new_content, str) else new_content
    fingerprint = hashlib.sha256(data).hexdigest()

    conn = get_db()
    conn.autocommit = True
    cur = conn.cursor()

    # Bump version and update metadata
    cur.execute("""
        UPDATE files
        SET file_size   = %s,
            fingerprint = %s,
            version     = version + 1,
            updated_at  = CURRENT_TIMESTAMP
        WHERE id = %s
        RETURNING version, storage_key
    """, (len(data), fingerprint, file_id))
    new_version, storage_key = cur.fetchone()

    # Overwrite in MinIO
    s3 = get_s3()
    s3.put_object(Bucket=BUCKET, Key=storage_key, Body=data)

    # Record the sync event
    cur.execute("""
        INSERT INTO sync_events (user_id, file_id, event_type)
        VALUES (%s, %s, 'updated')
    """, (owner_id, file_id))

    cur.close()
    conn.close()
    return new_version


# Alice updates report.txt from her laptop
new_ver = update_file(f1, alice_id, "Q1 sales were strong. Q2 projections added.")
print(f"💻 Alice's laptop updated report.txt → now version {new_ver}")

In [ ]:
# ── Alice's phone polls again ─────────────────────────────

print(f"📱 Alice's phone polls again (since={last_seen}):")
new_events = poll_changes(user_id=alice_id, since_event_id=last_seen)

if new_events:
    for e in new_events:
        print(f"   Event #{e['event_id']}: {e['event_type']} → {e['file_name']} (v{e['version']})")
    last_seen = new_events[-1]["event_id"]
    print(f"\n   ✅ Phone downloads the updated report.txt")
    print(f"   Phone now remembers: last_seen_event = {last_seen}")
else:
    print("   Nothing new!")

assert len(new_events) == 1, f"the laptop's edit should be the one new event, got {len(new_events)}"
assert new_events[0]["event_type"] == "updated"
assert new_events[0]["version"] == new_ver

print("\n💡 Polling is simple and reliable, but it has a downside:")
print("   The phone had to WAIT until the next poll interval to discover")
print("   the change. If the interval is 30s, the delay can be up to 30s.")
print("   Shorter intervals = faster sync but more wasted API calls.")

---

## 4️⃣ Push Notifications with Redis Pub/Sub

Polling works, but it's wasteful — most of the time nothing has changed, yet the client
keeps asking.  What if the server could **push** a notification the instant something changes?

In production, Dropbox uses **long-polling** and **WebSockets**. Here we'll use
**Redis Pub/Sub** as a simple stand-in.  The idea is the same:

```
Device A (laptop)          Server          Device B (phone)
    │                        │                  │
    │                        │◀── SUBSCRIBE ────│  phone listens on channel sync:1
    │── update file ────────▶│                  │
    │                        │── PUBLISH ──────▶│  "report.txt updated!"
    │                        │                  │── download new version
```

Each user gets their own channel: `sync:<user_id>`.  
When a file changes, we publish a JSON message to that channel.

In [ ]:
def publish_sync_notification(user_id, file_name, event_type, version):
    """
    Publish a sync notification to the user's Redis channel.
    In production this would go over WebSockets or SSE.
    """
    r = get_redis()
    channel = f"sync:{user_id}"
    message = json.dumps({
        "file_name": file_name,
        "event_type": event_type,
        "version": version,
        "timestamp": datetime.now().isoformat(),
    })
    r.publish(channel, message)
    return channel, message


# We'll collect messages received by the subscriber in this list
received_messages = []

def subscriber_loop(user_id, stop_event, ready_event):
    """
    Runs in a background thread. Listens on sync:<user_id> for messages.
    Signals `ready_event` once the SUBSCRIBE is confirmed by Redis, so the
    publisher never races ahead of the subscription. Stops when stop_event
    is set.
    """
    r = redis.Redis(**REDIS_CONFIG)  # each thread needs its own connection
    pubsub = r.pubsub()
    pubsub.subscribe(f"sync:{user_id}")

    for message in pubsub.listen():
        if message["type"] == "subscribe":
            # Redis has acknowledged the subscription — safe to publish now.
            ready_event.set()
            continue
        if stop_event.is_set():
            break
        if message["type"] == "message":
            data = json.loads(message["data"])
            received_messages.append(data)

    pubsub.unsubscribe()
    pubsub.close()


# ── Start the subscriber (simulating Alice's phone) ───────
stop_event = threading.Event()
ready_event = threading.Event()
received_messages.clear()

subscriber_thread = threading.Thread(
    target=subscriber_loop,
    args=(alice_id, stop_event, ready_event),
    daemon=True,
)
subscriber_thread.start()

# Block until Redis confirms the subscription rather than guessing with sleep().
assert ready_event.wait(timeout=5), "subscriber never subscribed to sync:1 — is Redis up?"

print("📱 Alice's phone is now listening on channel sync:1 ...")

In [ ]:
# ── Alice's laptop makes a change → server publishes ──────

new_ver = update_file(f1, alice_id, "Q1 sales were strong. Q2 projections added. Q3 targets set.")

# In a real system, the server would publish after the DB write.
channel, msg = publish_sync_notification(
    user_id=alice_id,
    file_name="report.txt",
    event_type="updated",
    version=new_ver,
)
print(f"💻 Laptop updated report.txt → v{new_ver}")
print(f"📡 Server published to {channel}")

# Wait for delivery instead of sleeping a fixed amount: poll the shared list
# until it fills, with a hard deadline so a broken Redis fails fast.
deadline = time.time() + 5
while not received_messages and time.time() < deadline:
    time.sleep(0.01)

# ── Check what the phone received ─────────────────────────
print(f"\n📱 Phone received {len(received_messages)} notification(s):")
for msg in received_messages:
    print(f"   → {msg['event_type']} {msg['file_name']} (v{msg['version']})")

assert len(received_messages) == 1, (
    f"push notification did not arrive: expected 1 message, got {len(received_messages)}"
)
assert received_messages[0]["version"] == new_ver

# Clean up subscriber thread
stop_event.set()
# Publish a dummy message to unblock the listen() loop
get_redis().publish(f"sync:{alice_id}", '"stop"')
subscriber_thread.join(timeout=2)

print("\n💡 The phone was notified INSTANTLY — no polling delay!")

### ⚠️ Why Pub/Sub alone isn't enough

Redis Pub/Sub is **fire-and-forget**: if the phone was offline when the message was
published, it **misses it forever**.  There's no replay.

| Approach | Pros | Cons |
|----------|------|------|
| Polling only | Simple, reliable, never misses events | Slow (up to 1 poll-interval delay), wastes bandwidth |
| Pub/Sub only | Instant notifications | Misses events when device is offline |

The solution? Use **both**. 👇

---

## 5️⃣ Hybrid Approach: Polling + Pub/Sub

Real sync clients combine both strategies:

- **Pub/Sub** (or WebSockets) for instant notifications while online
- **Polling** as a safety net — runs every N seconds to catch anything missed

```
┌────────────────────────────────────┐
│           SyncClient               │
│                                    │
│   pub/sub listener (real-time)     │
│        │                           │
│        ▼                           │
│   on_notification() ──▶ sync now   │
│                                    │
│   poll timer (every 30s)           │
│        │                           │
│        ▼                           │
│   poll_changes() ───▶ sync now     │
└────────────────────────────────────┘
```

Let's build a `SyncClient` class that does this.

In [ ]:
class SyncClient:
    """
    A simulated Dropbox sync client.
    - Listens on Redis pub/sub for instant push notifications.
    - Also polls the database periodically as a safety net.
    """

    def __init__(self, user_id, device_name, poll_interval=5):
        self.user_id = user_id
        self.device_name = device_name
        self.poll_interval = poll_interval
        self.last_seen_event = 0
        self.log = []  # record of all events this client processed
        self._stop = threading.Event()
        self._subscribed = threading.Event()  # set once Redis confirms SUBSCRIBE

    def _on_change(self, source, file_name, event_type, version):
        """Called whenever the client learns about a change."""
        entry = {
            "source": source,
            "file_name": file_name,
            "event_type": event_type,
            "version": version,
            "time": datetime.now().isoformat(),
        }
        self.log.append(entry)

    def _pubsub_loop(self):
        """Background thread: listen for push notifications."""
        r = redis.Redis(**REDIS_CONFIG)
        pubsub = r.pubsub()
        pubsub.subscribe(f"sync:{self.user_id}")

        for message in pubsub.listen():
            if message["type"] == "subscribe":
                self._subscribed.set()
                continue
            if self._stop.is_set():
                break
            if message["type"] == "message":
                try:
                    data = json.loads(message["data"])
                    if isinstance(data, dict):
                        self._on_change(
                            source="pubsub",
                            file_name=data["file_name"],
                            event_type=data["event_type"],
                            version=data["version"],
                        )
                except (json.JSONDecodeError, KeyError):
                    pass

        pubsub.unsubscribe()
        pubsub.close()

    def _poll_loop(self):
        """Background thread: periodic polling as safety net.

        start() already did the catch-up poll synchronously, so we wait out a
        full interval before the first one here. Polling immediately would race
        the pub/sub path for changes that happen right after start(), and the
        source of a change would become a coin flip.
        """
        while not self._stop.wait(self.poll_interval):
            events = poll_changes(self.user_id, self.last_seen_event)
            for e in events:
                self._on_change(
                    source="poll",
                    file_name=e["file_name"],
                    event_type=e["event_type"],
                    version=e["version"],
                )
                self.last_seen_event = e["event_id"]

    def start(self):
        """Start listening (pub/sub + polling)."""
        # Do an initial poll to catch up on anything we missed
        events = poll_changes(self.user_id, self.last_seen_event)
        for e in events:
            self.last_seen_event = e["event_id"]

        self._pubsub_thread = threading.Thread(target=self._pubsub_loop, daemon=True)
        self._poll_thread = threading.Thread(target=self._poll_loop, daemon=True)
        self._pubsub_thread.start()
        self._poll_thread.start()

        # Don't return until Redis has confirmed the subscription, otherwise a
        # publish that happens right after start() can be lost and the demo
        # below would flakily "miss" the push.
        assert self._subscribed.wait(timeout=5), "SyncClient never subscribed — is Redis up?"

    def stop(self):
        """Stop the client."""
        self._stop.set()
        # Unblock the pub/sub listen() loop
        get_redis().publish(f"sync:{self.user_id}", '"stop"')
        self._pubsub_thread.join(timeout=2)
        self._poll_thread.join(timeout=2)


print("✅ SyncClient class defined")
print("   It listens on pub/sub (instant) AND polls (safety net)")

In [ ]:
# ── Demo: SyncClient catches a push notification ──────────

phone_client = SyncClient(user_id=alice_id, device_name="Alice's phone", poll_interval=30)
phone_client.start()

# Alice edits on her laptop
new_ver = update_file(f2, alice_id, "Meeting notes from Monday. Action items added.")
publish_sync_notification(alice_id, "notes.txt", "updated", new_ver)
print(f"💻 Laptop updated notes.txt → v{new_ver}")

# Wait for the push rather than sleeping a fixed amount. The poll loop only
# fires every 30s, so anything that lands here came over pub/sub.
deadline = time.time() + 5
while not phone_client.log and time.time() < deadline:
    time.sleep(0.01)

print(f"\n📱 Phone SyncClient log ({len(phone_client.log)} events):")
for entry in phone_client.log:
    print(f"   [{entry['source']:>6}] {entry['event_type']} {entry['file_name']} (v{entry['version']})")

phone_client.stop()

assert phone_client.log, "SyncClient learned about nothing — the hybrid sync is broken"
assert phone_client.log[0]["source"] == "pubsub", (
    f"the change should arrive by push, not by the 30s poll; got {phone_client.log[0]['source']}"
)

print("\n💡 The pub/sub notification arrived instantly.")
print("   If it had been missed, the poll loop would catch it within 30s.")

---

## 6️⃣ Conflict Detection

Now for the tricky part. What happens when **two devices edit the same file**?

```
Alice's laptop                 Server (version=2)            Bob's desktop
    │                              │                              │
    │── "I edited report.txt       │                              │
    │    (my base was v2)" ───────▶│  ✅ v2 matches → accept      │
    │                              │     → set version=3           │
    │                              │                              │
    │                              │◀── "I edited report.txt      │
    │                              │     (my base was v2)" ───────│
    │                              │  ❌ v2 ≠ v3 → CONFLICT!      │
```

The key insight: we use the **version number** as an **optimistic lock**.

- When a client fetches a file, it notes the version (e.g., `v2`).
- When it submits an edit, it says: "I edited the file **based on version 2**."
- The server checks: is the current version still 2?
  - **Yes** → accept the edit, bump to v3.
  - **No** → someone else already changed it → **conflict detected**.

This is called **optimistic concurrency control** — we *optimistically* assume no
conflicts will happen, and only check at write time.

Let's implement it.

In [ ]:
class ConflictDetected(Exception):
    """Raised when a client's edit conflicts with another edit."""
    def __init__(self, file_name, client_version, server_version):
        self.file_name = file_name
        self.client_version = client_version
        self.server_version = server_version
        super().__init__(
            f"Conflict on '{file_name}': client has v{client_version}, "
            f"server has v{server_version}"
        )


def update_file_with_conflict_check(file_id, owner_id, new_content, expected_version):
    """
    Try to update a file, but ONLY if the current version matches what
    the client expects.  If someone else edited the file in the meantime,
    raise ConflictDetected.
    """
    import hashlib

    data = new_content.encode("utf-8") if isinstance(new_content, str) else new_content
    fingerprint = hashlib.sha256(data).hexdigest()

    conn = get_db()
    conn.autocommit = True
    cur = conn.cursor()

    # Optimistic lock: only update WHERE version = expected_version
    cur.execute("""
        UPDATE files
        SET file_size   = %s,
            fingerprint = %s,
            version     = version + 1,
            updated_at  = CURRENT_TIMESTAMP
        WHERE id = %s AND version = %s
        RETURNING version, file_name, storage_key
    """, (len(data), fingerprint, file_id, expected_version))

    result = cur.fetchone()

    if result is None:
        # The version didn't match — someone else edited the file!
        cur.execute("SELECT version, file_name FROM files WHERE id = %s", (file_id,))
        server_version, file_name = cur.fetchone()
        cur.close()
        conn.close()
        raise ConflictDetected(file_name, expected_version, server_version)

    new_version, file_name, storage_key = result

    # Write bytes to MinIO
    s3 = get_s3()
    s3.put_object(Bucket=BUCKET, Key=storage_key, Body=data)

    # Record sync event
    cur.execute("""
        INSERT INTO sync_events (user_id, file_id, event_type)
        VALUES (%s, %s, 'updated')
    """, (owner_id, file_id))

    cur.close()
    conn.close()
    return new_version


print("✅ update_file_with_conflict_check() defined")
print("   Uses optimistic concurrency control (version check)")

In [ ]:
# ── Demo: Two devices edit the same file ──────────────────

# First, let's see what version report.txt is at
conn = get_db()
cur = conn.cursor()
cur.execute("SELECT version FROM files WHERE id = %s", (f1,))
current_version = cur.fetchone()[0]
cur.close()
conn.close()
print(f"📄 report.txt is currently at version {current_version}")
print()

# Both Alice's laptop and phone think the file is at this version
laptop_version = current_version
phone_version = current_version

# Keep both bodies around — the resolution demos below need them.
laptop_content = "LAPTOP EDIT: Q1 strong, Q2 projections updated by laptop."
phone_content = "PHONE EDIT: Q1 strong, added notes from phone call."

# Alice's laptop edits first — this succeeds
print("💻 Alice's laptop submits edit (based on v{})...".format(laptop_version))
laptop_ver = None
try:
    laptop_ver = update_file_with_conflict_check(
        f1, alice_id, laptop_content, expected_version=laptop_version,
    )
    print(f"   ✅ Success! report.txt is now version {laptop_ver}")
except ConflictDetected as e:
    print(f"   ❌ {e}")

print()

# Alice's phone edits second — still thinks file is at old version!
print("📱 Alice's phone submits edit (based on v{})...".format(phone_version))
phone_conflict = None
try:
    new_ver = update_file_with_conflict_check(
        f1, alice_id, phone_content, expected_version=phone_version,
    )
    print(f"   ✅ Success! report.txt is now version {new_ver}")
except ConflictDetected as e:
    phone_conflict = e
    print(f"   ❌ CONFLICT DETECTED: {e}")
    print(f"   The phone's edit was based on an outdated version!")

print()
print("💡 The version column acts as a 'lock' that catches concurrent edits.")
print("   Now we need to decide: what do we DO about the conflict?")

# The whole section is worthless if the second write quietly succeeds.
assert laptop_ver == laptop_version + 1, (
    f"the first writer should win cleanly, expected v{laptop_version + 1}, got {laptop_ver}"
)
assert phone_conflict is not None, "the stale second write was accepted — optimistic locking is broken"
assert (phone_conflict.client_version, phone_conflict.server_version) == (phone_version, laptop_ver)

---

## 7️⃣ Conflict Resolution Strategies

When a conflict is detected, the system must resolve it. There are several strategies:

### Strategy 1: Last Write Wins (LWW)

The simplest approach: whoever writes **last** overwrites the other.  
We compare timestamps — the newer edit wins.

| Pros | Cons |
|------|------|
| Simple to implement | The "loser" edit is silently discarded |
| No user interaction needed | Data loss if both edits were important |

**Used by:** Amazon DynamoDB, Cassandra (for eventual consistency)

### Strategy 2: Keep Both Copies

Save the conflicting edit as a separate file, like:
`report (conflict copy - Alice's phone).txt`

The user sees both files and can manually merge them.

| Pros | Cons |
|------|------|
| No data loss | Clutters the file list |
| User decides how to merge | Requires manual intervention |

**Used by:** Dropbox, OneDrive, Syncthing

Let's implement both.

In [ ]:
def resolve_last_write_wins(file_id, new_content, editor_id):
    """
    Strategy 1: Last Write Wins
    Force-overwrite the file, ignoring the version check.
    The latest edit simply replaces whatever was there.
    """
    import hashlib

    data = new_content.encode("utf-8") if isinstance(new_content, str) else new_content
    fingerprint = hashlib.sha256(data).hexdigest()

    conn = get_db()
    conn.autocommit = True
    cur = conn.cursor()

    # Force update — no version check
    cur.execute("""
        UPDATE files
        SET file_size   = %s,
            fingerprint = %s,
            version     = version + 1,
            updated_at  = CURRENT_TIMESTAMP
        WHERE id = %s
        RETURNING version, file_name, storage_key
    """, (len(data), fingerprint, file_id))
    new_version, file_name, storage_key = cur.fetchone()

    s3 = get_s3()
    s3.put_object(Bucket=BUCKET, Key=storage_key, Body=data)

    cur.execute("""
        INSERT INTO sync_events (user_id, file_id, event_type)
        VALUES (%s, %s, 'updated')
    """, (editor_id, file_id))

    cur.close()
    conn.close()
    return new_version, file_name


# ── Demo: Last Write Wins ─────────────────────────────────
print("🏆 Strategy 1: Last Write Wins")
print("=" * 50)

ver, name = resolve_last_write_wins(f1, phone_content, editor_id=alice_id)
print(f"   Forced phone's edit onto {name} → now v{ver}")
print(f"   ⚠️  The laptop's edit is GONE. Simple, but risky!")

# Show the loss rather than asserting it away: the stored bytes are the
# phone's, and nothing anywhere still holds what the laptop wrote.
s3 = get_s3()
conn = get_db()
cur = conn.cursor()
cur.execute("SELECT storage_key FROM files WHERE id = %s", (f1,))
stored = s3.get_object(Bucket=BUCKET, Key=cur.fetchone()[0])["Body"].read().decode()
cur.close()
conn.close()
print(f"   Stored bytes now: \"{stored}\"")
assert stored == phone_content, "last-write-wins should have overwritten the file"
assert laptop_content not in stored, "the laptop's edit should be unrecoverable — that is the cost of LWW"

In [ ]:
def resolve_keep_both(original_file_id, conflict_content, editor_id, device_name):
    """
    Strategy 2: Keep Both Copies
    The original file stays unchanged.
    The conflicting edit is saved as a new file:
      'filename (conflict copy - device).ext'

    The copy is owned by the *original* file's owner, because that is where it
    belongs in the folder tree. If the losing editor was a collaborator rather
    than the owner we share it straight back to them — otherwise the person
    whose work we just "preserved" would never see it in their change feed,
    and "no data loss" would be a lie.
    """
    conn = get_db()
    conn.autocommit = True
    cur = conn.cursor()

    # Get the original file's name
    cur.execute("SELECT file_name, owner_id FROM files WHERE id = %s", (original_file_id,))
    original_name, owner_id = cur.fetchone()

    # Build the conflict copy name: "report.txt" → "report (conflict copy - phone).txt"
    if "." in original_name:
        base, ext = original_name.rsplit(".", 1)
        conflict_name = f"{base} (conflict copy - {device_name}).{ext}"
    else:
        conflict_name = f"{original_name} (conflict copy - {device_name})"

    cur.close()
    conn.close()

    # Create the conflict copy as a brand-new file
    conflict_file_id = create_file(owner_id, conflict_name, conflict_content)

    if editor_id != owner_id:
        conn = get_db()
        conn.autocommit = True
        cur = conn.cursor()
        cur.execute("""
            INSERT INTO shared_files (file_id, shared_with, permission)
            VALUES (%s, %s, 'write')
            ON CONFLICT (file_id, shared_with) DO UPDATE SET permission = 'write'
        """, (conflict_file_id, editor_id))
        cur.execute("""
            INSERT INTO sync_events (user_id, file_id, event_type)
            VALUES (%s, %s, 'shared')
        """, (editor_id, conflict_file_id))
        cur.close()
        conn.close()

    return conflict_file_id, conflict_name


# ── Demo: Keep Both Copies ────────────────────────────────
print("📋 Strategy 2: Keep Both Copies")
print("=" * 50)

# Strategy 1 just threw the laptop's edit away. Strategy 2 is what we should
# have done with it: park the losing edit next to the file instead of deleting it.
conflict_id, conflict_name = resolve_keep_both(
    original_file_id=f1,
    conflict_content=laptop_content,
    editor_id=alice_id,
    device_name="laptop",
)
print(f"   Original file (report.txt) keeps the phone's edit.")
print(f"   Conflict copy saved as: '{conflict_name}' (file_id={conflict_id})")
print(f"   ✅ No data loss! Alice can manually merge the two files.")

# Both edits must now be recoverable — that is the entire selling point.
conn = get_db()
cur = conn.cursor()
cur.execute("SELECT storage_key FROM files WHERE id = %s", (conflict_id,))
copy_key = cur.fetchone()[0]
cur.close()
conn.close()
copy_bytes = get_s3().get_object(Bucket=BUCKET, Key=copy_key)["Body"].read().decode()
assert copy_bytes == laptop_content, (
    f"the conflict copy should hold the losing edit, got \"{copy_bytes}\""
)

In [ ]:
# ── Let's see what Alice's file list looks like now ───────

conn = get_db()
cur = conn.cursor(cursor_factory=psycopg2.extras.RealDictCursor)
cur.execute("""
    SELECT id, file_name, version, file_size, status, updated_at
    FROM files
    WHERE owner_id = %s
    ORDER BY file_name
""", (alice_id,))
files = cur.fetchall()
cur.close()
conn.close()

print("📁 Alice's files:")
print(f"{'ID':>4}  {'File Name':<50} {'Ver':>4} {'Size':>6} {'Status':<10}")
print("-" * 85)
for f in files:
    print(f"{f['id']:>4}  {f['file_name']:<50} v{f['version']:>3} "
          f"{f['file_size']:>5}B {f['status']:<10}")

print("\n💡 The 'conflict copy' pattern is exactly what Dropbox does in real life!")
print("   You've probably seen files like 'document (John's conflicted copy).docx'")

### 🤔 Which strategy should you choose?

It depends on the use case:

| Strategy | Best for | Avoid when |
|----------|----------|------------|
| **Last Write Wins** | Settings files, caches, logs | Documents, spreadsheets |
| **Keep Both** | Documents, important data | Automated systems (too many copies) |
| **Merge** (not shown) | Code (git), text (CRDTs) | Binary files (images, videos) |

Dropbox defaults to **Keep Both** for most files because losing user data is worse
than having an extra file.

---

## 8️⃣ Simulating a Full Sync Cycle

Let's put it all together. Here's the scenario:

1. Alice creates `budget.txt` on her laptop
2. Bob gets write access (shared file)
3. Both Alice and Bob edit the file at the "same time"
4. The system detects the conflict
5. The conflict is resolved using the "keep both" strategy
6. Both devices sync and see the final state

This is the full lifecycle of a sync conflict.

In [ ]:
bob_id = 2

print("━" * 60)
print("  FULL SYNC CYCLE SIMULATION")
print("━" * 60)

# ── Step 1: Alice creates budget.txt ───────────────────────
print("\n📝 Step 1: Alice creates budget.txt on her laptop")
budget_id = create_file(alice_id, "budget.txt", "Total budget: $10,000")
print(f"   Created file_id={budget_id}, version=1")

# ── Step 2: Share with Bob ─────────────────────────────────
print("\n🤝 Step 2: Alice shares budget.txt with Bob (write access)")
conn = get_db()
conn.autocommit = True
cur = conn.cursor()
cur.execute("""
    INSERT INTO shared_files (file_id, shared_with, permission)
    VALUES (%s, %s, 'write')
""", (budget_id, bob_id))
cur.execute("""
    INSERT INTO sync_events (user_id, file_id, event_type)
    VALUES (%s, %s, 'shared')
""", (alice_id, budget_id))
cur.close()
conn.close()
print(f"   Bob now has write access to budget.txt")

In [ ]:
# ── Step 3: Both edit at the "same time" ───────────────────
print("\n✏️  Step 3: Both Alice and Bob edit budget.txt")
print("   (Both think the file is at version 1)")

alice_base_version = 1
bob_base_version = 1
alice_ver = None
bob_conflict_content = None

# Alice edits first — succeeds
print("\n   💻 Alice (laptop): 'Total budget: $10,000. Marketing: $3,000'")
try:
    alice_ver = update_file_with_conflict_check(
        budget_id, alice_id,
        "Total budget: $10,000. Marketing: $3,000",
        expected_version=alice_base_version,
    )
    print(f"   ✅ Alice's edit accepted → version {alice_ver}")
except ConflictDetected as e:
    print(f"   ❌ {e}")

# ── Step 4: Bob edits — conflict! ─────────────────────────
print("\n   🖥️  Bob (desktop): 'Total budget: $10,000. Engineering: $5,000'")
try:
    new_ver = update_file_with_conflict_check(
        budget_id, bob_id,
        "Total budget: $10,000. Engineering: $5,000",
        expected_version=bob_base_version,
    )
    print(f"   ✅ Bob's edit accepted → version {new_ver}")
except ConflictDetected as e:
    print(f"   ❌ CONFLICT! {e}")
    bob_conflict_content = "Total budget: $10,000. Engineering: $5,000"

# Two offline edits from the same base version MUST collide. If they don't,
# every cell below is resolving a conflict that never happened.
assert alice_ver == 2, f"Alice's edit should have taken the file to v2, got {alice_ver}"
assert bob_conflict_content is not None, (
    "Bob's stale edit was accepted silently — the sync engine just lost Alice's work"
)

In [ ]:
# ── Step 5: Resolve the conflict (keep both) ──────────────
print("\n🔧 Step 5: Resolve conflict using 'Keep Both' strategy")

conflict_id, conflict_name = resolve_keep_both(
    original_file_id=budget_id,
    conflict_content=bob_conflict_content,
    editor_id=bob_id,
    device_name="Bob's desktop",
)
print(f"   Original: budget.txt (Alice's version stays)")
print(f"   Conflict: {conflict_name} (Bob's version saved separately)")

# ── Step 6: Both devices sync ─────────────────────────────
print("\n🔄 Step 6: Both devices poll for the latest state")

for who, uid in [("Alice", alice_id), ("Bob", bob_id)]:
    feed = [e for e in poll_changes(uid, since_event_id=0)
            if e["file_name"].startswith("budget")]
    print(f"\n   📥 {who} → GET /changes?since=0 — {len(feed)} event(s):")
    print(f"   {'Event':>5}  {'File':<45} {'Type':<10} {'Ver'}")
    print("   " + "-" * 72)
    for e in feed:
        print(f"   {e['event_id']:>5}  {e['file_name']:<45} "
              f"{e['event_type']:<10} v{e['version']}")

# "Keep both" only counts as no-data-loss if the losing editor can actually
# reach the copy. Bob is not the owner, so the copy has to be shared back to him.
bob_files = {e["file_name"] for e in poll_changes(bob_id, since_event_id=0)}
alice_files = {e["file_name"] for e in poll_changes(alice_id, since_event_id=0)}
assert "budget.txt" in bob_files, f"Bob should see the file shared with him, saw {bob_files}"
assert conflict_name in bob_files, (
    f"Bob's own edit was parked somewhere he cannot see it; his feed is {bob_files}"
)
assert conflict_name in alice_files, "the owner should see the conflict copy too"

In [ ]:
# ── Final state: what does each file contain? ─────────────
print("\n📦 Final file contents in object storage:")
print("=" * 55)

conn = get_db()
cur = conn.cursor(cursor_factory=psycopg2.extras.RealDictCursor)
cur.execute("""
    SELECT file_name, storage_key, version
    FROM files
    WHERE file_name LIKE 'budget%%'
    ORDER BY file_name
""")
budget_files = cur.fetchall()
cur.close()
conn.close()

s3 = get_s3()
contents = {}
for bf in budget_files:
    obj = s3.get_object(Bucket=BUCKET, Key=bf["storage_key"])
    content = obj["Body"].read().decode("utf-8")
    contents[bf["file_name"]] = content
    print(f"\n   📄 {bf['file_name']} (v{bf['version']}):")
    print(f"      \"{content}\"")

# Neither edit may have been dropped on the floor.
assert "Marketing: $3,000" in contents["budget.txt"], "Alice's accepted edit is missing"
assert "Engineering: $5,000" in contents[conflict_name], "Bob's conflicting edit is missing"

print("\n" + "━" * 60)
print("  ✅ SYNC CYCLE COMPLETE")
print("━" * 60)
print("\n💡 Both edits are preserved. Alice and Bob can now review")
print("   the conflict copy and manually merge the changes.")

---

## 🧹 Cleanup

Remove the data we created during this notebook.

In [ ]:
# Clean up database
conn = get_db()
conn.autocommit = True
cur = conn.cursor()
cur.execute("DELETE FROM sync_events")
cur.execute("DELETE FROM shared_files")
cur.execute("DELETE FROM chunks")
cur.execute("DELETE FROM files")
cur.close()
conn.close()

# Clean up Redis
r = get_redis()
for key in r.scan_iter("sync:*"):
    r.delete(key)

# Clean up MinIO bucket contents
s3 = get_s3()
try:
    response = s3.list_objects_v2(Bucket=BUCKET)
    if "Contents" in response:
        for obj in response["Contents"]:
            s3.delete_object(Bucket=BUCKET, Key=obj["Key"])
        print(f"🗑️  Deleted {len(response['Contents'])} objects from MinIO")
except Exception:
    pass

print("✅ Cleanup complete — database, Redis, and MinIO are clean")

---

## 📝 Summary

In this notebook we built the **sync engine** of a Dropbox-like system. Here are the key takeaways:

| Concept | What we learned |
|---------|----------------|
| **Sync is hard** | Multiple devices + offline edits + network issues = conflict |
| **Event log** | The `sync_events` table records every mutation as an event |
| **Polling** | Client asks "what changed since event #N?" — simple but slow |
| **Pub/Sub push** | Server pushes changes instantly via Redis Pub/Sub — fast but unreliable |
| **Hybrid sync** | Combine both: pub/sub for speed, polling as safety net |
| **Conflict detection** | Use `version` column as an optimistic lock — reject stale edits |
| **Last Write Wins** | Simple, but silently discards one edit (data loss risk) |
| **Keep Both Copies** | Safe — saves a "conflict copy" file for the user to merge |

### 🔑 Key design principle

> **Never lose user data.** When in doubt, keep both versions and let the user decide.
> This is why Dropbox creates "conflicted copy" files.

### ➡️ Next notebook

In **Notebook 3: Deduplication**, we'll see how Dropbox avoids storing the same bytes twice
using content-based fingerprinting (file- and chunk-level), content-defined chunking, and reference counting.